In [1]:
import os
import fitz  # PyMuPDF for PDF text extraction
import gensim
from gensim import corpora
from gensim.models import CoherenceModel
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import nltk
import re
from collections import Counter
import pyLDAvis.gensim_models as gensimvis
import pyLDAvis


In [2]:
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Zviad\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Zviad\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [17]:
# Define PDF directory
pdf_folder = r"C:\Zviad_Thinkpad\Zviad\work\Healthcare data\Kaggle\Resume Dataset\data\data\Automobile"

In [18]:
# Extract text from PDFs
def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = " ".join(page.get_text() for page in doc)
    return text if text.strip() else "empty_document"

In [19]:
resume_stopwords = {
    "summary", "profile", "objective", "experience", "education", "skills",
    "certifications", "references", "managed", "developed", "led",
    "worked", "provided", "assisted", "collaborated", "cv", "resume",
    "applicant", "position", "role", "responsibilities", "employment",
    "year", "month", "date", "location", "address", "phone", "email"
}

In [20]:
# Minimal preprocessing for high-frequency analysis
def minimal_preprocessing(text):
    text = text.lower()
    text = re.sub(r'\W+', ' ', text)  # Remove punctuation
    tokens = word_tokenize(text)  # Tokenize words
    return tokens  # Return tokens for frequency analysis

In [21]:
# Collect PDFs and preprocess minimally
pdf_files = [os.path.join(pdf_folder, f) for f in os.listdir(pdf_folder) if f.endswith(".pdf")]
corpus = [minimal_preprocessing(extract_text_from_pdf(pdf)) for pdf in pdf_files]

In [22]:
# Extract high-frequency words dynamically
word_counts = Counter(word for doc in corpus for word in doc)
high_freq_terms = {word for word, count in word_counts.most_common(150)}  # Top 50 frequent words

In [23]:
# Merge industry-specific and standard stopwords
standard_stopwords = set(stopwords.words("english"))
filtered_stopwords = standard_stopwords.union(high_freq_terms).union(resume_stopwords)

In [24]:
# Final preprocessing with complete stopword list
def final_preprocessing(tokens):
    return [word for word in tokens if word.isalnum() and word not in filtered_stopwords]

In [25]:
corpus = [final_preprocessing(doc) for doc in corpus]

In [26]:
# Convert text into LDA format
dictionary = corpora.Dictionary(corpus)
doc_term_matrix = [dictionary.doc2bow(doc) for doc in corpus]

In [27]:
# Apply LDA
lda_model = gensim.models.LdaModel(doc_term_matrix, num_topics=10, id2word=dictionary, passes=10, alpha='auto', eta='auto')

In [28]:
# Print discovered topics
for idx, topic in enumerate(lda_model.print_topics()):
    print(f"Topic {idx}: {topic[1]}")

Topic 0: 0.006*"cim" + 0.005*"negotiate" + 0.004*"liability" + 0.004*"architecture" + 0.004*"web" + 0.003*"architect" + 0.003*"workflows" + 0.003*"evaluate" + 0.003*"assist" + 0.003*"results"
Topic 1: 0.009*"aircraft" + 0.007*"engineer" + 0.006*"contract" + 0.006*"suppliers" + 0.006*"engine" + 0.005*"airlines" + 0.005*"aviation" + 0.005*"parts" + 0.005*"site" + 0.005*"components"
Topic 2: 0.010*"mechanical" + 0.008*"yrs" + 0.007*"internet" + 0.006*"com" + 0.005*"club" + 0.005*"aaa" + 0.004*"projects" + 0.004*"online" + 0.004*"c" + 0.004*"modeling"
Topic 3: 0.005*"2010" + 0.005*"complaints" + 0.005*"accurately" + 0.005*"accordance" + 0.004*"established" + 0.004*"duties" + 0.004*"manner" + 0.004*"possible" + 0.004*"incoming" + 0.004*"march"
Topic 4: 0.007*"leader" + 0.006*"coaching" + 0.005*"executive" + 0.005*"events" + 0.005*"co" + 0.005*"weekly" + 0.005*"volume" + 0.005*"shift" + 0.005*"000" + 0.004*"meetings"
Topic 5: 0.006*"plan" + 0.005*"server" + 0.004*"engineering" + 0.004*"admin

In [29]:
# Visualize topics with pyLDAvis
vis = gensimvis.prepare(lda_model, doc_term_matrix, dictionary)
pyLDAvis.display(vis)

In [30]:
coherence_model = CoherenceModel(model=lda_model, texts=corpus, dictionary=dictionary, coherence='c_v')
coherence_score = coherence_model.get_coherence()

print(f"\nLDA Model Coherence Score: {coherence_score:.4f}")


LDA Model Coherence Score: 0.4401
